In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader

from dscript import FullyConnectedEmbed, ContactCNN, ModelInteraction
from dscript import PairedDataset
from dscript import collate_paired_sequences
from dscript import interaction_grad, interaction_eval

In [ ]:
spe = "yeast"

# data_dir = "ppi-data"
# train_file = os.path.join(data_dir, spe, "action/train_action_20.tsv")
# val_file = os.path.join(data_dir, spe, "action/test_action_20.tsv")
# epochs = 10

from google.colab import drive

drive.mount('/content/drive')
data_dir = "drive/MyDrive/ppi-data"
train_file = os.path.join(data_dir, spe, "action/train_action.tsv")
val_file = os.path.join(data_dir, spe, "action/test_action.tsv")
epochs = 50

embedding_h5 = os.path.join(data_dir, spe, "seq/pipr.embedding.h5")

input_dim = 13
embed_dim = 26

batch_size = 16
lr = 0.001

out_dir = "result/dscript"

seed = 1234
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

use_cuda = torch.cuda.is_available()

train_df = pd.read_csv(train_file, sep="\t", header=None)
train_dataset = PairedDataset(train_df[0].to_list(), train_df[1].to_list(),
                              train_df[2].to_list(), embedding_h5)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=collate_paired_sequences,
    shuffle=True,
)

val_df = pd.read_csv(val_file, sep="\t", header=None)
val_dataset = PairedDataset(val_df[0].to_list(), val_df[1].to_list(),
                            val_df[2].to_list(), embedding_h5)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=collate_paired_sequences
)


In [ ]:
embedding = FullyConnectedEmbed(input_dim, embed_dim)
contact = ContactCNN(embed_dim)
model = ModelInteraction(embedding, contact)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:

losses = []
val_losses = []

for epoch in range(epochs):
    batch_losses = []
    for x1, x2, y in train_loader:
        model.train()
        batch_loss, y_pred = interaction_grad(model, x1, x2, y, use_cuda)
        batch_losses.append(batch_loss)
        optimizer.step()
        optimizer.zero_grad()

    loss = np.mean(batch_losses)
    losses.append(loss)

    model.eval()
    with torch.no_grad():
        batch_losses = []
        n = 0
        accuracy_average = 0
        for x1, x2, y in val_loader:
            batch_loss, y_pred = interaction_eval(model, x1, x2, y, use_cuda)
            batch_losses.append(batch_loss)
            accuracy = accuracy_score(y, (y_pred > 0.5).int())
            accuracy_average = (accuracy_average * n + accuracy * x1.shape[0]) / (n + x1.shape[0])
            n = n + x1.shape[0]

        val_loss = np.mean(batch_losses)
        val_losses.append(val_loss)

    print(
        f"Epoch: {epoch + 1} -- loss: {loss}, val_loss: {val_loss}, accuracy: {accuracy_average:.4f}")